# SHIPP Stage 9B — Databricks AI Search index + semantic probe

`gold_listing_search_docs` → Vector Search endpoint `VS_ENDPOINT_NAME` → Delta Sync index `VS_INDEX_NAME`

| Decision | Value |
|---|---|
| Index type | Delta Sync, TRIGGERED (sync after every Gold MERGE — measured below) |
| Primary key | `listing_id` |
| Embedded column | `search_text`, embedded by `EMBEDDING_ENDPOINT` (Databricks-managed embeddings) |
| Proof | a query that is **not** the listing title returns the expected listing, and every result traces to Gold |

Creates the endpoint/index only if missing. **Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert.

In [0]:
%run ../common/shipp_silver_lib

In [0]:

import importlib
import config.tables as tables

importlib.invalidate_caches()
importlib.reload(tables)

print("Config loaded from:", tables.__file__)
print("VS_INDEX_NAME:", tables.VS_INDEX_NAME)

assert tables.VS_INDEX_NAME.endswith(
    "gold_listing_search_docs_index_v2"
), "STOP: Databricks is still loading the old index name."


## Config

In [0]:
from config.settings import (
    ROUTE_KEY_DECIMALS,
    SCORING,
    VISION_PROMPT_VERSION,
    VISION_ENDPOINT,
    VS_ENDPOINT_NAME,
    EMBEDDING_ENDPOINT,
    SEARCH_TEXT_COLUMN,
    SEARCH_RETURN_COLUMNS,
    RAG_PROBE_QUERY,
    RAG_PROBE_EXPECTED_LISTING,
    INDEX_SYNC_POLL_SECONDS,
    INDEX_SYNC_TIMEOUT_SECONDS,
)


import importlib
import config.tables as tables

importlib.invalidate_caches()
importlib.reload(tables)

VS_INDEX_NAME = tables.VS_INDEX_NAME
GOLD_LISTING_SEARCH_DOCS = tables.GOLD_LISTING_SEARCH_DOCS

assert VS_INDEX_NAME.endswith("_index_v2"), (
    f"Wrong AI Search index: {VS_INDEX_NAME}"
)


print("ROUTE_KEY_DECIMALS =", ROUTE_KEY_DECIMALS)
print("SCORING =", SCORING["version"])
print("VISION_PROMPT_VERSION =", VISION_PROMPT_VERSION)
print("VISION_ENDPOINT =", VISION_ENDPOINT)

print("VS_ENDPOINT_NAME =", VS_ENDPOINT_NAME)
print("VS_INDEX_NAME =", VS_INDEX_NAME)
print("EMBEDDING_ENDPOINT =", EMBEDDING_ENDPOINT)

print("SEARCH_TEXT_COLUMN =", SEARCH_TEXT_COLUMN)
print("RAG_PROBE_QUERY =", RAG_PROBE_QUERY)
print("RAG_PROBE_EXPECTED_LISTING =", RAG_PROBE_EXPECTED_LISTING)

print("INDEX_SYNC_POLL_SECONDS =", INDEX_SYNC_POLL_SECONDS)
print("INDEX_SYNC_TIMEOUT_SECONDS =", INDEX_SYNC_TIMEOUT_SECONDS)

print("\nCONFIG CHECK: PASS")

In [0]:
# Reload the rag modules so edits to rag/*.py always take effect (same rule as shipp_silver_lib).
for _name in ["rag.text", "rag.vision", "rag.content", "rag.search_docs", "rag.index",
              "rag.retrieval", "rag.semantic", "rag.evaluation"]:
    importlib.reload(importlib.import_module(_name))

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("RAG modules reloaded from", REPO_ROOT)

In [0]:
try:
    _ep = w.serving_endpoints.get(EMBEDDING_ENDPOINT)
    print("Embedding endpoint:", _ep.name, "| ready:", _ep.state.ready if _ep.state else None)
except Exception as e:
    _names = sorted(s.name for s in w.serving_endpoints.list())
    raise AssertionError(
        f"FAILED: embedding endpoint {EMBEDDING_ENDPOINT!r} not found ({type(e).__name__}). "
        f"Set EMBEDDING_ENDPOINT in config/settings.py to one of: {_names}"
    )

## Step 1 — Endpoint + index

In [0]:

from rag import index
from rag.retrieval import search_listing_context

state = index.ensure_endpoint(w, VS_ENDPOINT_NAME)

assert state == "ONLINE", (
    f"FAILED: Vector Search endpoint {VS_ENDPOINT_NAME} "
    f"is {state}, not ONLINE"
)

columns_to_sync = SEARCH_RETURN_COLUMNS + [
    "vision_status",
    "document_updated_at"
]

created = index.ensure_delta_sync_index(
    w,
    VS_INDEX_NAME,
    VS_ENDPOINT_NAME,
    GOLD_LISTING_SEARCH_DOCS,
    primary_key="listing_id",
    text_column=SEARCH_TEXT_COLUMN,
    embedding_endpoint=EMBEDDING_ENDPOINT,
    columns_to_sync=columns_to_sync,
)


In [0]:

idx = w.vector_search_indexes.get_index(
    index_name=VS_INDEX_NAME
)

pipeline_id = (
    idx.delta_sync_index_spec.pipeline_id
    if idx.delta_sync_index_spec
    else None
)

print("Delta Sync pipeline ID:", pipeline_id)

if pipeline_id:
    p = w.pipelines.get(pipeline_id=pipeline_id)
    print("Pipeline name:", p.name)
    print("Pipeline state:", p.state)
    print("Pipeline health:", p.health)
else:
    print("Pipeline ID not available yet.")


In [0]:
from rag import index
from rag.retrieval import search_listing_context

state = index.ensure_endpoint(w, VS_ENDPOINT_NAME)
assert state == "ONLINE", f"FAILED: Vector Search endpoint {VS_ENDPOINT_NAME} is {state}, not ONLINE"

pipeline_id = (
    idx.delta_sync_index_spec.pipeline_id
    if idx.delta_sync_index_spec
    else None
)

print("Delta Sync pipeline ID:", pipeline_id or "Not available yet")

columns_to_sync = SEARCH_RETURN_COLUMNS + ["vision_status", "document_updated_at"]
created = index.ensure_delta_sync_index(
    w, VS_INDEX_NAME, VS_ENDPOINT_NAME, GOLD_LISTING_SEARCH_DOCS,
    primary_key="listing_id", text_column=SEARCH_TEXT_COLUMN,
    embedding_endpoint=EMBEDDING_ENDPOINT, columns_to_sync=columns_to_sync,
)

pipeline_id = (
    idx.delta_sync_index_spec.pipeline_id
    if idx.delta_sync_index_spec
    else None
)

print("Delta Sync pipeline ID:", pipeline_id or "Not available yet")



## Step 2 — Sync and measure how long Gold → searchable takes

In [0]:
from databricks.sdk import WorkspaceClient
PIPELINE_ID = pipeline_id
w = WorkspaceClient()

print("=== VECTOR SEARCH ENDPOINT ===")
ep = w.vector_search_endpoints.get_endpoint(
    endpoint_name=VS_ENDPOINT_NAME
)
print("name :", ep.name)
print("state:", ep.endpoint_status.state if ep.endpoint_status else None)

print("\n=== SHIPP INDEX ===")
idx = w.vector_search_indexes.get_index(
    index_name=VS_INDEX_NAME
)

print("index :", idx.name)
print("endpoint:", idx.endpoint_name)

if idx.status:
    print("ready             :", idx.status.ready)
    print("indexed_row_count :", idx.status.indexed_row_count)
    print("message           :", idx.status.message)

if idx.delta_sync_index_spec:
    print("pipeline_id       :", idx.delta_sync_index_spec.pipeline_id)
    print("pipeline_type     :", idx.delta_sync_index_spec.pipeline_type)
    print("source_table      :", idx.delta_sync_index_spec.source_table)

In [0]:
pipeline_id = (
    idx.delta_sync_index_spec.pipeline_id
    if idx.delta_sync_index_spec
    else None
)

if pipeline_id:
    p = w.pipelines.get(pipeline_id=pipeline_id)

    print("\n=== DELTA SYNC PIPELINE ===")
    print("Pipeline ID:", pipeline_id)
    print("State:", p.state)
    print("Health:", p.health)

    for update in (p.latest_updates or [])[:3]:
        print("Update state:", update.state)
        print("Created:", update.creation_time)
else:
    print("Delta Sync pipeline ID not available yet.")

In [0]:
pipeline_id = idx.delta_sync_index_spec.pipeline_id

p = w.pipelines.get(pipeline_id=PIPELINE_ID)

print("=== DELTA SYNC PIPELINE ===")
print("pipeline_id :", PIPELINE_ID)
print("name        :", p.name)
print("state       :", p.state)
print("health      :", p.health)
print("cause       :", p.cause)
print("cluster_id  :", p.cluster_id)

print("\n=== LATEST UPDATES ===")

if p.latest_updates:
    for u in p.latest_updates[:5]:
        print(
            "update_id =", u.update_id,
            "| state =", u.state,
            "| created =", u.creation_time
        )
else:
    print("No updates reported yet.")

In [0]:

idx = w.vector_search_indexes.get_index(
    index_name=VS_INDEX_NAME
)

print("=== INDEX STATUS ===")
print("Ready:", idx.status.ready)
print("Indexed rows:", idx.status.indexed_row_count)
print("Message:", idx.status.message)

pipeline_id = idx.delta_sync_index_spec.pipeline_id
p = w.pipelines.get(pipeline_id=pipeline_id)

print("\n=== DELTA SYNC PIPELINE ===")
print("Pipeline ID:", pipeline_id)
print("State:", p.state)
print("Health:", p.health)

for update in (p.latest_updates or [])[:3]:
    print("Update state:", update.state)
    print("Created:", update.creation_time)


In [0]:
expected_rows = spark.table(GOLD_LISTING_SEARCH_DOCS).count()
if not created:
    try:
        index.trigger_sync(w, VS_INDEX_NAME)
        print("Sync triggered.")
    except Exception as e:  # a sync already running is fine; we wait for it below
        print(f"Sync not triggered ({type(e).__name__}: {str(e)[:150]}) — waiting for the running one.")
sync_seconds = index.wait_until_synced(w, VS_INDEX_NAME, expected_rows)
print(f"PASS — index holds {expected_rows} docs. Gold → searchable: {sync_seconds}s")
print(index.index_status(w, VS_INDEX_NAME))

## Step 3 — Semantic probe with traceability

In [0]:
t0 = time.time()
results = search_listing_context(RAG_PROBE_QUERY, k=3, workspace_client=w)
probe_ms = int((time.time() - t0) * 1000)

print(f"Query: {RAG_PROBE_QUERY!r}  ({probe_ms} ms)")
for r in results:
    print(f"  {(r['score'] or 0):.4f}  {r['listing_id']:<20} {r['title']}  | {r['snippet'][:120]}")

gold_ids = {r.listing_id for r in spark.table(GOLD_LISTING_SEARCH_DOCS).select("listing_id").collect()}
assert results, "FAILED: the probe returned nothing."
assert all(r["listing_id"] in gold_ids for r in results), "FAILED: a result does not trace to Gold"
assert RAG_PROBE_EXPECTED_LISTING in [r["listing_id"] for r in results], (
    f"FAILED: {RAG_PROBE_EXPECTED_LISTING} not in the top 3 — check it is AVAILABLE and in Gold."
)
print("PASS — semantic retrieval returns the expected listing, every result traceable by listing_id.")

summary = {"index": VS_INDEX_NAME, "docs": expected_rows, "sync_seconds": sync_seconds,
           "probe_latency_ms": probe_ms, "status": "PASS"}
print(summary)
dbutils.notebook.exit(json.dumps(summary))